In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('mutual_fund_data.csv')
print("Scheme Details Data:")
print(df.head())

Scheme Details Data:
   Scheme_Code                                 Scheme_Name  \
0       100033  Aditya Birla Sun Life Large & Mid Cap Fund   
1       100034  Aditya Birla Sun Life Large & Mid Cap Fund   
2       100037           Aditya Birla Sun Life Income Fund   
3       100038           Aditya Birla Sun Life Income Fund   
4       100039           Aditya Birla Sun Life Income Fund   

                                 AMC Scheme_Type  \
0  Aditya Birla Sun Life AMC Limited  Open Ended   
1  Aditya Birla Sun Life AMC Limited  Open Ended   
2  Aditya Birla Sun Life AMC Limited  Open Ended   
3  Aditya Birla Sun Life AMC Limited  Open Ended   
4  Aditya Birla Sun Life AMC Limited  Open Ended   

                              Scheme_Category  \
0        Equity Scheme - Large & Mid Cap Fund   
1        Equity Scheme - Large & Mid Cap Fund   
2  Debt Scheme - Medium to Long Duration Fund   
3  Debt Scheme - Medium to Long Duration Fund   
4  Debt Scheme - Medium to Long Duration Fund   

In [5]:
df.shape

(16109, 16)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16109 entries, 0 to 16108
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Scheme_Code                              16109 non-null  int64  
 1   Scheme_Name                              16109 non-null  object 
 2   AMC                                      16109 non-null  object 
 3   Scheme_Type                              16109 non-null  object 
 4   Scheme_Category                          16109 non-null  object 
 5   Scheme_NAV_Name                          16109 non-null  object 
 6   Scheme_Min_Amt                           16079 non-null  object 
 7   NAV                                      14073 non-null  float64
 8   Latest_NAV_Date                          14073 non-null  object 
 9   Average_AUM_Cr                           7929 non-null   float64
 10  AAUM_Quarter                             7929 

In [7]:
print(df['Scheme_Category'].unique())

['Equity Scheme - Large & Mid Cap Fund'
 'Debt Scheme - Medium to Long Duration Fund' 'Debt Scheme - Liquid Fund'
 'Debt Scheme - Gilt Fund' 'Equity Scheme - Sectoral/ Thematic'
 'Debt Scheme - Medium Duration Fund' 'Equity Scheme - Flexi Cap Fund'
 'Hybrid Scheme - Aggressive Hybrid Fund'
 'Debt Scheme - Money Market Fund'
 'Hybrid Scheme - Dynamic Asset Allocation or Balanced Advantage'
 'Equity Scheme - ELSS' 'Equity Scheme - Small Cap Fund'
 'Equity Scheme - Large Cap Fund' 'Equity Scheme - Value Fund' 'ELSS'
 'Debt Scheme - Long Duration Fund' 'Balanced'
 'Equity Scheme - Mid Cap Fund' 'Other Scheme - Index Funds'
 'Debt Scheme - Dynamic Bond' 'Debt Scheme - Low Duration Fund'
 'Debt Scheme - Corporate Bond Fund'
 'Solution Oriented Scheme - Retirement Fund' 'Gilt'
 'Hybrid Scheme - Conservative Hybrid Fund'
 'Equity Scheme - Multi Cap Fund' 'Growth'
 'Debt Scheme - Ultra Short Duration Fund'
 'Solution Oriented Scheme - Children s Fund'
 'Debt Scheme - Banking and PSU Fund' 'Debt

In [16]:
# Use the exact string from your list
overnight_funds_df = df[df['Scheme_Category'] == 'Debt Scheme - Overnight Fund']

# You can now see your new, filtered data
print(overnight_funds_df.size)


4944


In [9]:
df_nav_history = pd.read_parquet('mutual_fund_nav_history.parquet')
df_nav_history.head()

,Date,NAV
Scheme_Code,,
100033,2006-04-03,116.61
100033,2013-10-31,164.70
100033,2006-08-21,104.73
100033,2007-08-07,136.34
100033,2014-04-01,188.16


In [13]:
overnight_scheme_codes = overnight_funds_df['Scheme_Code'].unique()
df_nav_history = df_nav_history.reset_index() 

# Now filter the data
print("Filtering historical data for overnight funds...")
df_overnight_data = df_nav_history[
    df_nav_history['Scheme_Code'].isin(overnight_scheme_codes)
].copy() # Use .copy() to avoid SettingWithCopyWarning


# --- Step 3: Data Cleaning and Preprocessing ---
# Now this code will work because 'Date' is a column

print("Converting 'Date' column to datetime...")
df_overnight_data['Date'] = pd.to_datetime(df_overnight_data['Date'])

print("Sorting data by Scheme_Code and Date...")
df_overnight_data = df_overnight_data.sort_values(
    by=['Scheme_Code', 'Date'], 
    ascending=True
)

print("\nChecking for missing values:")
print(df_overnight_data.isnull().sum())

print("Setting 'Date' as the index for time-series analysis...")
df_overnight_data.set_index('Date', inplace=True)

print("\nCleaned and Sorted Data (Head):")
print(df_overnight_data.head())

Filtering historical data for overnight funds...
Converting 'Date' column to datetime...
Sorting data by Scheme_Code and Date...

Checking for missing values:
index          0
Scheme_Code    0
Date           0
NAV            0
dtype: int64
Setting 'Date' as the index for time-series analysis...

Cleaned and Sorted Data (Head):
             index  Scheme_Code      NAV
Date                                    
2006-04-03  985076       100813  10.3536
2006-04-04  986775       100813  10.3586
2006-04-05  986060       100813  10.3643
2006-04-07  987986       100813  10.3679
2006-04-10  986591       100813  10.3724


In [14]:
import pandas as pd
import sys

# --- Step 1: Define the 5-year time window ---
# Using today's date: November 3, 2025
today = pd.to_datetime('2025-11-03')
start_date = today - pd.DateOffset(years=5)
print(f"Filtering data from {start_date.date()} to {today.date()}...")

# --- Step 2: Re-filter from your *original* clean DataFrame ---
# This creates a new df_last_five_years that is guaranteed
# to have the 'Date' as its index.
try:
    df_last_five_years = df_overnight_data[df_overnight_data.index >= start_date].copy()
except Exception as e:
    print(f"Error filtering df_overnight_data: {e}")
    print("Please make sure 'df_overnight_data' is your clean, main DataFrame with 'Date' as the index.")
    sys.exit()

# --- Step 3: Reset the index ---
# This moves 'Date' from the index to be a regular column.
df_last_five_years.reset_index(inplace=True)
# Now you have a 'Date' column and a 'Scheme_Code' column.

# --- Step 4: Create the 'Scheme_Name' lookup table ---
try:
    df_schemes = pd.read_csv('mutual_fund_data.csv')
    df_name_lookup = df_schemes[['Scheme_Code', 'Scheme_Name']].drop_duplicates()
except FileNotFoundError:
    print("ERROR: 'mutual_fund_data.csv' not found. Please download it.")
    sys.exit()

# --- Step 5: Merge to add the 'Scheme_Name' ---
df_last_five_years = pd.merge(
    df_last_five_years,
    df_name_lookup,
    on='Scheme_Code',
    how='left'
)
# This drops the column named 'index' and saves the change
df_last_five_years = df_last_five_years.drop(columns=['index'])

# --- Done ---
print("--- 'index' column has been dropped. ---")
print(df_last_five_years.head())
# --- Done ---
print("\n--- Success! Your DataFrame is fixed. ---")
print(df_last_five_years.head())

print("\n--- Columns in your new DataFrame ---")
print(df_last_five_years.info())

Filtering data from 2020-11-03 to 2025-11-03...
--- 'index' column has been dropped. ---
        Date  Scheme_Code        NAV           Scheme_Name
0 2020-11-03       100813  1470.7953  UTI - Overnight Fund
1 2020-11-04       100813  1470.9100  UTI - Overnight Fund
2 2020-11-05       100813  1471.0236  UTI - Overnight Fund
3 2020-11-08       100813  1471.3704  UTI - Overnight Fund
4 2020-11-09       100813  1471.4856  UTI - Overnight Fund

--- Success! Your DataFrame is fixed. ---
        Date  Scheme_Code        NAV           Scheme_Name
0 2020-11-03       100813  1470.7953  UTI - Overnight Fund
1 2020-11-04       100813  1470.9100  UTI - Overnight Fund
2 2020-11-05       100813  1471.0236  UTI - Overnight Fund
3 2020-11-08       100813  1471.3704  UTI - Overnight Fund
4 2020-11-09       100813  1471.4856  UTI - Overnight Fund

--- Columns in your new DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352948 entries, 0 to 352947
Data columns (total 4 columns):
 #   Column

In [17]:
import pandas as pd
import sys

# Define a string variable for your output filename
output_filename = 'Group11_Data.xlsx'  # <-- The filename must be a string

try:
    with pd.ExcelWriter(output_filename) as writer:
        
        df_last_five_years.to_excel(
            writer, 
            sheet_name='Last_5_Years', # Changed sheet name for clarity
            index=False # Keep the 'Date' index
        )
        
        overnight_funds_df.to_excel( 
            writer, 
            sheet_name='Scheme_Details', 
            index=False # Don't need the 0,1,2 index
        )
        

    print(f"--- Success! ---")
    print(f"Saved DataFrames to '{output_filename}' in separate sheets.")

except ImportError:
    print("\n--- ERROR ---")
    print("You must install 'openpyxl' to write Excel files.")
    print("Run this command in your terminal: pip install openpyxl")
except NameError as e:
    print(f"\n--- NameError ---")
    print(f"A variable was not defined: {e}")
    print("Make sure 'df_last_five_years', 'df_scheme_details', and 'df_overnight_data' exist.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Success! ---
Saved DataFrames to 'Group11_Data.xlsx' in separate sheets.


In [11]:
df_scheme_details= pd.read_csv('Scheme_details.csv')

In [12]:
df_scheme_details.head(10)

,Scheme_Code,Scheme_Name,AMC,Scheme_Type,Scheme_Category,Scheme_NAV_Name,1_Year_Return,Standard_Deviation,Beta,Alpha,Scheme_Min_Amt,NAV,Latest_NAV_Date,Average_AUM_Cr,AAUM_Quarter,ISIN_Div_Payout/Growth,ISIN_Div_Reinvestment,ISIN_Div_Payout/Growth/Div_Reinvestment,Launch_Date,Closure_Date
0,100814,UTI - Overnight Fund,UTI Asset Mgmt. Co. Ltd.,Open Ended,Debt Scheme - Overnight Fund,UTI - Overnight Fund - Regular Plan - Growth ...,0.057802,0.001306,0.229025,0.013515,Rs.10000/,3570.8136,2025-11-02,1043.9006,July - September 2025,INF789F01604,-,INF789F01604,1999-08-23,NaN
1,101206,SBI OVERNIGHT FUND,SBI Funds Management Limited,Open Ended,Debt Scheme - Overnight Fund,SBI OVERNIGHT FUND - REGULAR PLAN - GROWTH,0.057365,0.001310,0.317450,0.017887,10000,4232.9785,2025-11-02,2985.8952,July - September 2025,INF200K01LQ9,-,INF200K01LQ9,2005-07-25,NaN
2,101996,HDFC Overnight Fund,HDFC Asset Management Company Limited,Open Ended,Debt Scheme - Overnight Fund,HDFC Overnight Fund - Growth Option,0.056900,0.001331,0.273756,0.015636,REFER SID,3871.7535,2025-11-02,3113.9393,July - September 2025,INF179KB1HS3,-,INF179KB1HS3,2002-02-06,NaN
3,119110,HDFC Overnight Fund,HDFC Asset Management Company Limited,Open Ended,Debt Scheme - Overnight Fund,HDFC Overnight Fund - Growth Option - Direct Plan,0.057718,0.001354,0.278519,0.016446,REFER SID,3910.3782,2025-11-02,8474.7980,July - September 2025,INF179KB1HT1,-,INF179KB1HT1,2002-02-06,NaN
4,119833,SBI OVERNIGHT FUND,SBI Funds Management Limited,Open Ended,Debt Scheme - Overnight Fund,SBI OVERNIGHT FUND - DIRECT PLAN - GROWTH,0.058067,0.001326,0.319846,0.018523,10000,4289.4116,2025-11-02,22089.7119,July - September 2025,INF200K01TK5,-,INF200K01TK5,2005-07-25,NaN
5,120785,UTI - Overnight Fund,UTI Asset Mgmt. Co. Ltd.,Open Ended,Debt Scheme - Overnight Fund,UTI - Overnight Fund - Direct Plan - Growth Op...,0.058315,0.001318,0.231052,0.013979,Rs.10000/,3610.1589,2025-11-02,4371.9985,July - September 2025,INF789FB1S71,-,INF789FB1S71,1999-08-23,NaN
6,145481,ADITYA BIRLA SUN LIFE OVERNIGHT FUND,Aditya Birla Sun Life AMC Limited,Open Ended,Debt Scheme - Overnight Fund,ADITYA BIRLA SUNLIFE OVERNIGHT FUND-REGULAR PL...,0.057177,0.001419,0.276074,0.015741,500,1415.0560,2025-11-02,1721.4663,July - September 2025,INF209KB1ZC3,-,INF209KB1ZC3,2018-10-30,NaN
7,145486,ADITYA BIRLA SUN LIFE OVERNIGHT FUND,Aditya Birla Sun Life AMC Limited,Open Ended,Debt Scheme - Overnight Fund,ADITYA BIRLA SUN LIFE OVERNIGHT FUND-DIRECT PL...,0.058273,0.001445,0.278701,0.016708,500,1426.4855,2025-11-02,7109.5230,July - September 2025,INF209KB1ZH2,-,INF209KB1ZH2,2018-10-30,NaN
8,145535,ICICI Prudential Overnight Fund,ICICI Prudential Asset Management Company Limited,Open Ended,Debt Scheme - Overnight Fund,ICICI Prudential Overnight Fund - Growth,0.057539,0.001352,0.296851,0.017044,Rs. 100/- (plus in multiple of Re. 1),1412.8846,2025-11-02,2704.0480,July - September 2025,INF109KC17E3,-,INF109KC17E3,2018-11-14,NaN
9,145536,ICICI Prudential Overnight Fund,ICICI Prudential Asset Management Company Limited,Open Ended,Debt Scheme - Overnight Fund,ICICI Prudential Overnight Fund - Direct Plan ...,0.058206,0.001369,0.300780,0.017715,Rs. 100/- (plus in multiple of Re. 1),1421.1632,2025-11-02,9407.4978,July - September 2025,INF109KC11G1,-,INF109KC11G1,2018-11-14,NaN


In [15]:
df_scheme_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 20 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Scheme_Code                              76 non-null     int64  
 1   Scheme_Name                              76 non-null     object 
 2   AMC                                      76 non-null     object 
 3   Scheme_Type                              76 non-null     object 
 4   Scheme_Category                          76 non-null     object 
 5   Scheme_NAV_Name                          76 non-null     object 
 6   1_Year_Return                            74 non-null     float64
 7   Standard_Deviation                       74 non-null     float64
 8   Beta                                     74 non-null     float64
 9   Alpha                                    74 non-null     float64
 10  Scheme_Min_Amt                           76 non-null

In [31]:
df_scheme_details.drop('ISIN_Div_Payout/Growth',axis=1,inplace=True)

In [32]:
import pandas as pd
import sys

# Define a string variable for your output filename
output_filename = 'Scheme_details_final.xlsx'  # <-- The filename must be a string

try:
    with pd.ExcelWriter(output_filename) as writer:
        
       df_scheme_details.to_excel(
            writer, 
            index=False # Keep the 'Date' index
        )

    print(f"--- Success! ---")
    print(f"Saved DataFrames to '{output_filename}' in separate sheets.")

except ImportError:
    print("\n--- ERROR ---")
    print("You must install 'openpyxl' to write Excel files.")
    print("Run this command in your terminal: pip install openpyxl")
except NameError as e:
    print(f"\n--- NameError ---")
    print(f"A variable was not defined: {e}")
    print("Make sure 'df_last_five_years', 'df_scheme_details', and 'df_overnight_data' exist.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Success! ---
Saved DataFrames to 'Scheme_details_final.xlsx' in separate sheets.


In [36]:
df_scheme_details_clean=df_scheme_details.dropna()
df_scheme_details_clean.shape

(73, 16)